In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset
import torch.nn.utils.prune as prune
from torchvision import datasets, transforms, models
import os
import random
import numpy as np
import copy
import time

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

In [2]:
def prune_model(model, pruning_amount=0.3, pruning_method=prune.L1Unstructured):
    model = copy.deepcopy(model)
    parameters_to_prune = []
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            parameters_to_prune.append((module, 'weight'))
        elif isinstance(module, nn.Linear):
            parameters_to_prune.append((module, 'weight'))
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=pruning_method,
        amount=pruning_amount
    )
    
    for module, _ in parameters_to_prune:
        prune.remove(module, 'weight')
    
    return model

In [3]:
def train(model, train_loader, val_loader, optimizer, criterion, num_epoch):
    print('Training....')
    start = time.time()
    
    for epoch in range(num_epoch):
        total = 0
        correct = 0
        for i, data in enumerate(train_loader, 1):
            images, labels = data
    
            images = images.to(device)
            labels = labels.to(device)
    
            optimizer.zero_grad()    
            outputs = model(images)
    
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
            loss = criterion(outputs, labels)
            # if (i % 100 == 0):
            #     print('Epoch: {} Batch: {} loss: {}'.format(epoch, i, loss.item()))
    
            loss.backward()
            optimizer.step()
        evaluate(model, val_loader)
    
    print('Training Completed in: {} secs'.format(time.time()-start))
    print('Training accuracy: {} %'.format((correct/total)*100))

In [4]:
def evaluate(model, test_loader):
    total = 0
    correct = 0
    for i, data in enumerate(test_loader, 1):
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    print('Accuracy: {} %'.format((correct/total)*100))
    return (correct/total)*100

# MNIST

In [5]:
hyperparams = {
    "batch_size": 256,
}

In [6]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

mnist_train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
mnist_val_loader = DataLoader(val_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
mnist_test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)

## Resnet-18 pruning

In [7]:
hyperparams = {
    "num_epochs": 10,
    "batch_size": 256,
    "lr": 4e-5,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [14]:
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(512, 10, bias=True)

# for param in model.parameters():
#     param.requires_grad = False

# for param in model.fc.parameters():
#     param.requires_grad = True

In [15]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

11181642

In [16]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"])

In [17]:
train(model, mnist_train_loader, mnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 97.55 %
Accuracy: 98.44166666666668 %
Accuracy: 98.65 %
Accuracy: 98.69166666666666 %
Accuracy: 98.73333333333333 %
Accuracy: 98.825 %
Accuracy: 98.86666666666667 %
Accuracy: 98.7 %
Accuracy: 98.73333333333333 %
Accuracy: 98.88333333333334 %
Training Completed in: 256.5457663536072 secs
Training accuracy: 99.95208333333333 %


In [18]:
start_time = time.time()
evaluate(model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 99.00999999999999 %
--- 3.9249401092529297 seconds ---


In [45]:
pruned_model = prune_model(model, pruning_amount=0.75)

In [46]:
start_time = time.time()
evaluate(pruned_model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 97.31 %
--- 3.887023687362671 seconds ---


In [16]:
11181642 * (1 - 0.75)

2795410.5

## Resnext50 pruning

In [22]:
hyperparams = {
    "num_epochs": 10,
    "batch_size": 256,
    "lr": 1e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [23]:
model = models.resnext50_32x4d(weights='DEFAULT')
model.fc = nn.Linear(2048, 10, bias=True)

In [24]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

23000394

In [25]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"], weight_decay=hyperparams["weight_decay"])

In [26]:
train(model, mnist_train_loader, mnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 96.61666666666666 %
Accuracy: 97.925 %
Accuracy: 98.075 %
Accuracy: 98.28333333333333 %
Accuracy: 98.43333333333332 %
Accuracy: 98.51666666666667 %
Accuracy: 98.44166666666668 %
Accuracy: 98.49166666666666 %
Accuracy: 98.49166666666666 %
Accuracy: 98.71666666666667 %
Training Completed in: 341.2189395427704 secs
Training accuracy: 99.70625 %


In [27]:
start_time = time.time()
evaluate(model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 98.7 %
--- 4.5626490116119385 seconds ---


In [34]:
pruned_model = prune_model(model, pruning_amount=0.75)

In [35]:
start_time = time.time()
evaluate(pruned_model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 97.72 %
--- 4.432665586471558 seconds ---


In [36]:
pytorch_total_params * (1 - 0.75)

5750098.5

## MNASNet-1 pruning

In [42]:
hyperparams = {
    "num_epochs": 10,
    "batch_size": 256,
    "lr": 4e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [43]:
model = models.mnasnet1_0(weights='DEFAULT')
model.classifier[1] = nn.Linear(1280, 10, bias=True)

In [44]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

3115122

In [45]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"], weight_decay=hyperparams["weight_decay"])

In [46]:
train(model, mnist_train_loader, mnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 97.41666666666666 %
Accuracy: 97.98333333333333 %
Accuracy: 98.09166666666667 %
Accuracy: 98.39166666666667 %
Accuracy: 98.58333333333333 %
Accuracy: 98.45833333333334 %
Accuracy: 98.66666666666667 %
Accuracy: 98.55000000000001 %
Accuracy: 98.425 %
Accuracy: 98.425 %
Training Completed in: 281.1548798084259 secs
Training accuracy: 99.38749999999999 %


In [48]:
start_time = time.time()
evaluate(model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 98.47 %
--- 3.824052333831787 seconds ---


In [63]:
pruned_model = prune_model(model, pruning_amount=0.825)

In [64]:
start_time = time.time()
evaluate(pruned_model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 97.39999999999999 %
--- 3.8713834285736084 seconds ---


In [65]:
pytorch_total_params * (1 - 0.825)

545146.3500000001

## EfficientNet-V2-Small pruning

In [74]:
hyperparams = {
    "num_epochs": 16,
    "batch_size": 256,
    "lr": 2e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [75]:
model = models.efficientnet_v2_s(weights='DEFAULT')
model.classifier[1] = nn.Linear(1280, 10, bias=True)

In [76]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

20190298

In [77]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"], weight_decay=hyperparams["weight_decay"])

In [78]:
train(model, mnist_train_loader, mnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 97.32499999999999 %
Accuracy: 97.95833333333334 %
Accuracy: 98.48333333333333 %
Accuracy: 98.65 %
Accuracy: 98.775 %
Accuracy: 99.01666666666667 %
Accuracy: 98.81666666666666 %
Accuracy: 99.10833333333333 %
Accuracy: 99.00833333333333 %
Accuracy: 98.9 %
Accuracy: 99.09166666666667 %
Accuracy: 99.05833333333334 %
Accuracy: 99.25833333333334 %
Accuracy: 99.075 %
Accuracy: 99.25833333333334 %
Accuracy: 99.08333333333333 %
Training Completed in: 706.253351688385 secs
Training accuracy: 99.68333333333334 %


In [79]:
start_time = time.time()
evaluate(model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 99.17 %
--- 4.914014101028442 seconds ---


In [94]:
pruned_model = prune_model(model, pruning_amount=0.91)

In [95]:
start_time = time.time()
evaluate(pruned_model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 97.65 %
--- 4.847860813140869 seconds ---


In [96]:
pytorch_total_params * (1 - 0.91)

1817126.8199999994

## MobileNet-V3-Small pruning

In [220]:
hyperparams = {
    "num_epochs": 16,
    "batch_size": 256,
    "lr": 8e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [221]:
model = models.mobilenet_v3_small(weights='DEFAULT')
model.classifier[3] = nn.Linear(1024, 10, bias=True)

In [222]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

1528106

In [223]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"], weight_decay=hyperparams["weight_decay"])

In [224]:
train(model, mnist_train_loader, mnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 95.94166666666666 %
Accuracy: 97.25833333333334 %
Accuracy: 97.66666666666667 %
Accuracy: 97.74166666666667 %
Accuracy: 97.89166666666667 %
Accuracy: 97.875 %
Accuracy: 98.1 %
Accuracy: 98.18333333333334 %
Accuracy: 97.96666666666667 %
Accuracy: 97.49166666666666 %
Accuracy: 98.14166666666667 %
Accuracy: 98.09166666666667 %
Accuracy: 98.15 %
Accuracy: 98.46666666666667 %
Accuracy: 98.41666666666666 %
Accuracy: 98.19166666666666 %
Training Completed in: 466.17599296569824 secs
Training accuracy: 98.97916666666666 %


In [225]:
start_time = time.time()
evaluate(model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 98.47 %
--- 4.026180982589722 seconds ---


In [226]:
pruned_model = prune_model(model, pruning_amount=0.937)

In [227]:
start_time = time.time()
evaluate(pruned_model, mnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 97.28 %
--- 3.993122100830078 seconds ---


In [137]:
pytorch_total_params * (1 - 0.937)

96270.67799999991

# FashionMNIST

In [143]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

batch_size = 64

fmnist_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
fmnist_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, generator=g)
fmnist_test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, generator=g)

## Resnet-18 pruning

In [69]:
hyperparams = {
    "num_epochs": 10,
    "batch_size": 256,
    "lr": 4e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [70]:
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(512, 10, bias=True)

In [ ]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

In [72]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"])

In [73]:
train(model, fmnist_train_loader, fmnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 87.11666666666666 %
Accuracy: 88.39166666666667 %
Accuracy: 89.44166666666666 %
Accuracy: 89.01666666666667 %
Accuracy: 89.4 %
Accuracy: 90.54166666666667 %
Accuracy: 89.60833333333333 %
Accuracy: 90.10000000000001 %
Accuracy: 90.46666666666667 %
Accuracy: 90.00833333333334 %
Training Completed in: 223.7557590007782 secs
Training accuracy: 95.275 %


In [74]:
start_time = time.time()
evaluate(model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 90.39 %
--- 2.1512482166290283 seconds ---


In [79]:
pruned_model = prune_model(model, pruning_amount=0.86)

In [80]:
start_time = time.time()
evaluate(pruned_model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 89.3 %
--- 2.1610019207000732 seconds ---


In [81]:
pytorch_total_params * (1 - 0.86)

1564551.8

## Resnext50 pruning

In [126]:
hyperparams = {
    "num_epochs": 20,
    "batch_size": 256,
    "lr": 5e-4,    
    "task_type": "classification",
    "weight_decay": 1e-4
}

name = ", ".join(
    f"{key}: {value.__class__.__name__ if key == 'metric' else value}"
    for key, value in hyperparams.items()
)

name

'num_epochs: 20, batch_size: 256, lr: 0.0005, task_type: classification, weight_decay: 0.0001'

In [127]:
model = models.resnext50_32x4d()
model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.fc = nn.Linear(2048, 10, bias=True)

In [128]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

22994122

In [129]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"])

In [130]:
train(model, fmnist_train_loader, fmnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 83.45 %
Accuracy: 85.70833333333333 %
Accuracy: 86.47500000000001 %
Accuracy: 86.96666666666667 %
Accuracy: 86.5 %
Accuracy: 87.10833333333333 %
Accuracy: 88.94999999999999 %
Accuracy: 88.33333333333333 %
Accuracy: 88.89166666666667 %
Accuracy: 89.09166666666667 %
Accuracy: 89.10833333333333 %
Accuracy: 89.31666666666666 %
Accuracy: 89.03333333333333 %
Accuracy: 87.00833333333333 %
Accuracy: 90.2 %
Accuracy: 90.03333333333333 %
Accuracy: 89.90833333333333 %
Accuracy: 90.225 %
Accuracy: 89.60833333333333 %
Accuracy: 90.30833333333334 %
Training Completed in: 986.5796263217926 secs
Training accuracy: 95.90833333333333 %


In [131]:
start_time = time.time()
evaluate(model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 89.74 %
--- 3.6487042903900146 seconds ---


In [136]:
pruned_model = prune_model(model, pruning_amount=0.5)

In [137]:
start_time = time.time()
evaluate(pruned_model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 89.01 %
--- 3.640805959701538 seconds ---


In [ ]:
pytorch_total_params * (1 - 0.86)

## MobileNet-V3-Small pruning

In [202]:
hyperparams = {
    "num_epochs": 10,
    "batch_size": 256,
    "lr": 5e-4,    
    "task_type": "classification",
    "weight_decay": 1e-3
}

In [203]:
model = models.mobilenet_v3_small(weights='DEFAULT')
model.classifier[3] = nn.Linear(1024, 10, bias=True)

In [204]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

1528106

In [205]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"], weight_decay=hyperparams["weight_decay"])

In [206]:
train(model, fmnist_train_loader, fmnist_val_loader, optimizer, criterion, hyperparams["num_epochs"])

Training....
Accuracy: 84.55 %
Accuracy: 87.04166666666666 %
Accuracy: 88.08333333333334 %
Accuracy: 88.70833333333333 %
Accuracy: 89.31666666666666 %
Accuracy: 89.68333333333334 %
Accuracy: 90.16666666666666 %
Accuracy: 90.19166666666666 %
Accuracy: 90.46666666666667 %
Accuracy: 90.70833333333333 %
Training Completed in: 526.8218274116516 secs
Training accuracy: 91.31041666666667 %


In [207]:
start_time = time.time()
evaluate(model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 90.11 %
--- 5.289144277572632 seconds ---


In [216]:
pruned_model = prune_model(model, pruning_amount=0.925)

In [217]:
start_time = time.time()
evaluate(pruned_model, fmnist_test_loader)
print("--- %s seconds ---" % (time.time() - start_time))

Accuracy: 89.29 %
--- 5.313025951385498 seconds ---


In [218]:
pytorch_total_params * (1 - 0.925)

114607.94999999994